In [1]:
from pathlib import Path

import polars as pl

project_path = Path.cwd() / "output" / "project"

In [2]:
cnpt_name = "white_blood_cell_count".lower().replace(" ", "_").replace("/", "-")
cnpt_pattern = "(51301//White Blood Cells)"
size = 4157284

In [3]:
cnpt_directory = project_path / "workspace" / "concept" / cnpt_name / "1.0.0"

cnpt_paths = [str(p) for p in cnpt_directory.iterdir() if p.is_file()]

cnpt =  pl.scan_parquet(cnpt_paths[0])
cnpt = cnpt.with_columns(pl.col("time").dt.replace_time_zone("UTC"))
cnpt = cnpt.collect()

In [4]:
len(cnpt) == size

True

In [5]:
cnpt

subject_id,time,code,numeric_value,text_value,dataset,table
i64,"datetime[μs, UTC]",str,f32,str,str,str
10000032,2180-03-23 11:51:00 UTC,"""white_blood_cell_count//K/uL""",3.0,"""3.0""","""mimic-iv""","""labevents"""
10000032,2180-05-06 22:25:00 UTC,"""white_blood_cell_count//K/uL""",5.0,"""5.0""","""mimic-iv""","""labevents"""
10000032,2180-05-07 05:05:00 UTC,"""white_blood_cell_count//K/uL""",4.2,"""4.2""","""mimic-iv""","""labevents"""
10000032,2180-06-22 11:15:00 UTC,"""white_blood_cell_count//K/uL""",5.1,"""5.1""","""mimic-iv""","""labevents"""
10000032,2180-06-26 16:10:00 UTC,"""white_blood_cell_count//K/uL""",6.6,"""6.6""","""mimic-iv""","""labevents"""
…,…,…,…,…,…,…
19999987,2145-11-04 10:40:00 UTC,"""white_blood_cell_count//K/uL""",11.6,"""11.6""","""mimic-iv""","""labevents"""
19999987,2145-11-05 06:10:00 UTC,"""white_blood_cell_count//K/uL""",10.0,"""10.0""","""mimic-iv""","""labevents"""
19999987,2145-11-06 10:07:00 UTC,"""white_blood_cell_count//K/uL""",5.9,"""5.9""","""mimic-iv""","""labevents"""


In [6]:
labevent = pl.scan_parquet(project_path / "workspace" / "extraction" / "mimic-iv" / "3.1" / "labevents" / "result.parquet")
data = labevent.filter(pl.col("code").str.contains(cnpt_pattern)).collect()

In [7]:
data.sort("time")

subject_id,time,code,numeric_value,text_value,hadm_id,stay_id
i64,datetime[μs],str,f32,str,str,str
16904137,2105-01-19 12:04:00,"""mimic-iv//3.1//labevents//resu…",7.2,"""7.2""",null,"""-1"""
16904137,2105-02-02 11:10:00,"""mimic-iv//3.1//labevents//resu…",8.9,"""8.9""",null,"""-1"""
16904137,2105-04-06 10:20:00,"""mimic-iv//3.1//labevents//resu…",7.7,"""7.7""",null,"""-1"""
16904137,2105-10-04 17:27:00,"""mimic-iv//3.1//labevents//resu…",9.6,"""9.6""",null,"""-1"""
16904137,2105-10-05 01:41:00,"""mimic-iv//3.1//labevents//resu…",11.5,"""11.5""","""21081215""","""-1"""
…,…,…,…,…,…,…
16573705,2215-01-01 04:17:00,"""mimic-iv//3.1//labevents//resu…",4.7,"""4.7""",null,"""-1"""
16573705,2215-01-01 11:42:00,"""mimic-iv//3.1//labevents//resu…",4.8,"""4.8""",null,"""-1"""
16573705,2215-01-02 02:26:00,"""mimic-iv//3.1//labevents//resu…",5.5,"""5.5""",null,"""-1"""


In [8]:
for e in data.unique("code")["code"]:
    print(e)

mimic-iv//3.1//labevents//result//51301//White Blood Cells//K/uL
mimic-iv//3.1//labevents//result//51301//White Blood Cells


In [9]:
ricu = pl.read_parquet("./data/cnpt_labevents.parquet")
ricu

FileNotFoundError: No such file or directory (os error 2): ./data/cnpt_labevents.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'


In [ ]:
#ricu.sort("charttime_origin")

In [ ]:
ricu.join(cnpt, left_on=["subject_id", "charttime"], right_on=["subject_id", "time"], how="anti")


labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
i32,i32,i32,i32,i32,str,"datetime[μs, UTC]","datetime[μs, UTC]",str,f64,str,f64,f64,str,str,str
154,10000032,22595853,86271148,50971,null,2180-05-07 05:05:00 UTC,2180-05-07 07:03:00 UTC,"""4.5""",4.5,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
181,10000032,null,19543630,50971,"""P85UQ1""",2180-06-03 12:00:00 UTC,2180-06-03 13:04:00 UTC,"""3.3""",3.3,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
190,10000032,null,58691952,50971,"""P69FQC""",2180-06-03 12:00:00 UTC,2180-06-03 13:04:00 UTC,"""3.4""",3.4,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
403,10000032,29079034,55621508,50971,null,2180-07-23 21:45:00 UTC,2180-07-23 22:30:00 UTC,"""4.7""",4.7,"""mEq/L""",3.3,5.1,null,"""STAT""",null
451,10000032,29079034,66433308,50971,null,2180-07-25 04:45:00 UTC,2180-07-25 07:44:00 UTC,"""5.2""",5.2,"""mEq/L""",3.3,5.1,"""abnormal""","""ROUTINE""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
159057396,19999987,23865745,34834852,50971,null,2145-11-04 05:01:00 UTC,2145-11-04 05:51:00 UTC,"""___""",4.1,"""mEq/L""",3.3,5.1,null,"""ROUTINE""","""HEMOLYSIS FALSELY ELEVATES K.."""
159057461,19999987,23865745,62584889,50971,null,2145-11-05 06:10:00 UTC,2145-11-05 07:02:00 UTC,"""3.6""",3.6,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null
159057486,19999987,23865745,45769962,50971,null,2145-11-06 10:07:00 UTC,2145-11-06 11:19:00 UTC,"""3.7""",3.7,"""mEq/L""",3.3,5.1,null,"""ROUTINE""",null


In [ ]:
cnpt.join(ricu, left_on=["subject_id", "time"], right_on=["subject_id", "charttime"], how="anti")

subject_id,time,code,numeric_value,text_value,dataset,table
i64,"datetime[μs, UTC]",str,f32,str,str,str
10000117,2174-09-22 12:26:00 UTC,"""albumin//g/dL""",4.7,"""4.7""","""mimic-iv""","""labevents"""
10000117,2175-01-27 16:15:00 UTC,"""albumin//g/dL""",4.7,"""4.7""","""mimic-iv""","""labevents"""
10000117,2176-02-08 22:00:00 UTC,"""albumin//g/dL""",4.9,"""4.9""","""mimic-iv""","""labevents"""
10000117,2176-09-08 10:00:00 UTC,"""albumin//g/dL""",4.9,"""4.9""","""mimic-iv""","""labevents"""
10000117,2178-06-12 15:19:00 UTC,"""albumin//g/dL""",4.8,"""4.8""","""mimic-iv""","""labevents"""
…,…,…,…,…,…,…
19999784,2119-06-27 05:40:00 UTC,"""albumin//g/dL""",3.7,"""3.7""","""mimic-iv""","""labevents"""
19999784,2119-07-07 17:30:00 UTC,"""albumin//g/dL""",4.3,"""4.3""","""mimic-iv""","""labevents"""
19999784,2123-04-12 12:28:00 UTC,"""albumin//g/dL""",4.0,"""4.0""","""mimic-iv""","""labevents"""


In [ ]:
# ricu.join(cnpt, left_on=["subject_id", "charttime", "valuenum"], right_on=["subject_id", "time", "numeric_value"], how="anti")

In [ ]:
# cnpt.join(ricu, left_on=["subject_id", "time", "numeric_value"], right_on=["subject_id", "charttime", "valuenum"], how="anti")